# Castle5000: VLM duyệt nhãn (Qwen2.5-VL) trên Kaggle / Colab

Notebook này KHÔNG cần ảnh gốc. Nó nhận gói kết quả đã có (`castle_results.zip`, tạo bằng `python main.py bundle pack`), cho VLM duyệt ảnh hạt giống và gợi ý của tầng người và từng vùng cố định, rồi đóng gói lại.

**Chuẩn bị:** đưa `labeling_code.zip` (hoặc thư mục `labeling/`) và `castle_results.zip` lên một Dataset (Kaggle) hoặc Google Drive (Colab). Bật GPU và Internet. Model Qwen2.5-VL-7B (khoảng 16 GB) được tải về ở lần chạy đầu.

Tiến độ nằm trong `WORK`: chạy lại sau khi bị ngắt sẽ không hỏi lại những crop VLM đã trả lời.

## 1. Cấu hình

In [ ]:
import os, sys, glob, shutil, zipfile, subprocess
from pathlib import Path

ON_KAGGLE = os.path.isdir("/kaggle/working")
ON_COLAB = not ON_KAGGLE and ("google.colab" in sys.modules or os.path.isdir("/content"))

CODE_ZIP = None       # None = tự tìm labeling_code.zip hoặc thư mục labeling/
RESULTS_ZIP = None    # None = tự tìm castle_results.zip hoặc thư mục đã giải nén
ROUNDS = 3            # số vòng: VLM duyệt -> gom cụm lại -> gợi ý
VLM_LIMIT = None      # trần lệnh gọi VLM cho MỖI profile; đặt số nhỏ (ví dụ 40) để thử nhanh
STEPS = "fill" # "fill,merge" = VLM gán nhãn trực tiếp mọi crop còn trống (sau khi đã chạy vlm)
PROFILES = "people" # None = người + mọi vùng; hoặc ví dụ "roi-fridge" (gói tạo bằng `bundle pack --profiles roi-fridge`)
BACKEND = "qwen_local"

if ON_KAGGLE:
    # WORK nằm ngoài /kaggle/working để hàng chục nghìn file crop không thành "output" của notebook
    # (tải về rất chậm). Chỉ file zip kết quả được ghi vào /kaggle/working. Muốn giữ tiến độ giữa các
    # phiên tương tác có bật Persistence thì đổi thành Path("/kaggle/working/castle").
    SEARCH, WORK = "/kaggle/input", Path("/kaggle/temp/castle")
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SEARCH, WORK = "/content/drive/MyDrive", Path("/content/drive/MyDrive/castle_work")
else:
    SEARCH, WORK = os.getcwd(), Path(os.getcwd()) / "castle_work"
WORK.mkdir(parents=True, exist_ok=True)
CODE_DIR = Path("/kaggle/working/labeling" if ON_KAGGLE else "/content/labeling" if ON_COLAB else WORK / "labeling")
print("Kaggle:", ON_KAGGLE, "| Colab:", ON_COLAB, "| WORK:", WORK)

## 2. Mã nguồn

In [ ]:
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
if CODE_ZIP is None:
    found = sorted(glob.glob(f"{SEARCH}/**/labeling_code.zip", recursive=True))
    CODE_ZIP = found[-1] if found else None
if CODE_ZIP:
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR.parent)
    source = CODE_ZIP
else:
    found = sorted(glob.glob(f"{SEARCH}/**/labeling/main.py", recursive=True))
    assert found, f"Không thấy labeling_code.zip hay labeling/main.py dưới {SEARCH}. Đặt CODE_ZIP bằng tay."
    source = os.path.dirname(found[-1])
    shutil.copytree(source, CODE_DIR)
assert (CODE_DIR / "main.py").exists(), f"Không có labeling/main.py trong {source}"
print("Mã nguồn:", CODE_DIR, "<-", source)

## 3. Thư viện và GPU

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements-cloud.txt")], check=True)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GB")
assert torch.cuda.is_available(), "Cần GPU cho bước VLM."

## 4. Nạp gói kết quả vào WORK

In [ ]:
os.environ["LABELING_OUTPUT_ROOT"] = str(WORK)
os.environ["LABELING_DATA_ROOT"] = str(WORK / "data")
os.environ["LABELING_INPUT_IMAGES"] = str(WORK / "no_images")     # bước vlm không đọc ảnh gốc
(WORK / "no_images").mkdir(exist_ok=True)
os.environ["PYTHONIOENCODING"] = "utf-8"

def run(*args):
    cmd = [sys.executable, "main.py", *[str(a) for a in args]]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    assert proc.wait() == 0, f"Lệnh thất bại: {' '.join(map(str, args))}"

if any((WORK / "output_jsons").glob("adl_*/approved.json")):
    print("WORK đã có kết quả: giữ nguyên để tiếp tục tiến độ VLM.")
else:
    if RESULTS_ZIP is None:
        found = sorted(glob.glob(f"{SEARCH}/**/castle_results.zip", recursive=True))
        RESULTS_ZIP = found[-1] if found else None
    if RESULTS_ZIP:
        run("bundle", "unpack", RESULTS_ZIP)
    else:
        # Kaggle tự giải nén file zip trong Dataset: tìm cây data/ + output_jsons/ đã bung.
        found = sorted(glob.glob(f"{SEARCH}/**/output_jsons/adl_*/approved.json", recursive=True))
        assert found, f"Không thấy castle_results.zip hay output_jsons/adl_* dưới {SEARCH}. Đặt RESULTS_ZIP bằng tay."
        root = Path(found[-1]).parents[2]
        for name in ("data", "output_jsons"):
            shutil.copytree(root / name, WORK / name, dirs_exist_ok=True)
        print("Kết quả:", WORK, "<-", root)
run("adl", "status", "--profile", (PROFILES or "people").split(",")[0])

## 5. VLM duyệt nhãn

Với mỗi ảnh hạt giống và gợi ý, VLM tự chọn một lớp (không được cho biết nhãn máy đã gợi ý). Trùng thì nhận nhãn; khác thì nhận lớp của VLM và đánh dấu để soát; "không lớp nào" thì loại. Sau mỗi vòng, các cụm được gom lại và gợi ý mới được sinh ra.

In [ ]:
args = ["pipeline", "--steps", STEPS, "--rounds", ROUNDS, "--backend", BACKEND]
if VLM_LIMIT:
    args += ["--vlm-limit", VLM_LIMIT]
if PROFILES:
    args += ["--profiles", PROFILES]
try:
    run(*args)
except AssertionError as e:
    # Vẫn đóng gói ở bước sau: câu trả lời VLM đã có nằm trong vlm_reviews.jsonl, không mất.
    print("LỖI:", e, "-> vẫn đóng gói phần đã làm")


## 6. Đóng gói

In [ ]:
RESULT_ZIP = (Path("/kaggle/working") if ON_KAGGLE else WORK) / "castle_results_vlm.zip"
run("bundle", "pack", RESULT_ZIP, "--light")
print("Tải file này về:", RESULT_ZIP, f"({RESULT_ZIP.stat().st_size / 1e6:.0f} MB)")

Ở máy local, trong `labeling/`:

```bash
python main.py bundle unpack duong_dan/castle_results_vlm.zip
python main.py adl export --profile people --what vlm
```

Gói `--light` chỉ chứa trạng thái các vòng lặp và bản ghi gộp (không kèm crop và embedding, vì máy local đã có). Lệnh thứ hai dựng trang soát lại những nhãn VLM đã quyết, chỗ VLM đổi nhãn xếp trước.

In [ ]:
import json
for store in sorted((WORK / "output_jsons").glob("adl_*")):
    approved = json.load(open(store / "approved.json", encoding="utf-8")) if (store / "approved.json").exists() else {}
    by = {}
    for r in approved.values():
        key = r.get("source") + ("/" + r["vlm"] if r.get("vlm") else "")
        by[key] = by.get(key, 0) + 1
    rejected = json.load(open(store / "rejected.json", encoding="utf-8")) if (store / "rejected.json").exists() else {}
    print(f"{store.name:26s} {by} | bị loại: {len(rejected)}")